In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
from ipywidgets import widgets

from latent_dynamics.dayang.activations import extract_activations
from latent_dynamics.dayang.data import DATASET_REGISTRY, load_dataset_from_spec
from latent_dynamics.dayang.model import MODEL_REGISTRY, load_model_and_tokenizer
from latent_dynamics.dayang.projections import compute_layerwise_pca, plot_layerwise_pca, plot_layerwise_pca_ratio
from latent_dynamics.dayang.scoring import compute_layerwise_score, plot_layerwise_score
from latent_dynamics.dayang.widgets import ActivationsExtractorWidget, ActivationsSelectorWidget

torch.set_grad_enabled(False)

print(f"Available models:{'\n - '.join([''] + list(MODEL_REGISTRY))}")
print(f"Available datasets:{'\n - '.join([''] + list(DATASET_REGISTRY))}")

# Manual analysis

In [ ]:
dataset = load_dataset_from_spec("xstest", max_samples=200)

In [ ]:
model, tokenizer = load_model_and_tokenizer("gemma3_270m")

In [ ]:
activations = extract_activations(model, tokenizer, dataset)
activations.metadata

## Projection analysis

In [ ]:
pcas = compute_layerwise_pca(activations, pool_method="last")
plot_layerwise_pca_ratio(pcas)

In [ ]:
plot_layerwise_pca(activations, pcas, pool_method="last")

## Scoring analysis

In [ ]:
scorers = compute_layerwise_score(activations, method="difference_in_mean")

In [ ]:
plot_layerwise_score(activations, scorers, pool_method="last")

# Interactive analysis

## Projection analysis

In [ ]:
def analyze_layerwise_pca():
    state = {"models": {}}

    # Define layer 1 widgets for extracting activations
    layer1 = ActivationsExtractorWidget(
        search_path="../activations",
        dataset="xstest",
    )

    # Define layer 2 widgets for analyzing activations
    w_train_selector = ActivationsSelectorWidget()
    w_eval_selector = ActivationsSelectorWidget()
    btn_analyze = widgets.Button(description="Analyze", button_style="primary", disabled=True)
    out_analyze = widgets.Output()

    col_train = widgets.VBox([widgets.HTML("<b>PCA</b>"), w_train_selector])
    col_eval = widgets.VBox([widgets.HTML("<b>Visualization</b>"), w_eval_selector])
    layer2 = widgets.VBox(
        [
            widgets.HBox([col_train, col_eval]),
            btn_analyze,
            out_analyze,
        ]
    )

    # Register handlers for layer 1
    def update_layer2_widgets(*args):
        activations = layer1.activations

        if activations is None:
            btn_analyze.disabled = True
        else:
            w_train_selector.set_activations(activations)
            w_eval_selector.set_activations(activations)
            btn_analyze.disabled = False
        out_analyze.clear_output()

    layer1.observe(update_layer2_widgets, names="value")

    # Register handlers for layer 2
    def do_plot(*args):
        activations = layer1.activations
        if activations is None:
            return

        with out_analyze:
            out_analyze.clear_output(wait=True)

            # Select samples training
            traub_sample_ids = w_train_selector.samples
            train_activations = activations.select(traub_sample_ids) if traub_sample_ids else activations
            # Train and cache model
            cache_key = (
                tuple(sorted(traub_sample_ids)),
                w_train_selector.pool_method,
                w_train_selector.exclude_bos,
                w_train_selector.exclude_special_tokens,
            )
            if cache_key not in state["models"]:
                state["models"][cache_key] = compute_layerwise_pca(
                    train_activations,
                    pool_method=w_train_selector.pool_method,
                    exclude_bos=w_train_selector.exclude_bos,
                    exclude_special_tokens=w_train_selector.exclude_special_tokens,
                )
            # Plot PCA explained variance ratio
            plot_layerwise_pca_ratio(state["models"][cache_key])

            # Select samples for evaluation
            eval_sample_ids = w_eval_selector.samples
            eval_activations = activations.select(eval_sample_ids) if eval_sample_ids else activations
            # Plot evaluation
            plot_layerwise_pca(
                eval_activations,
                state["models"][cache_key],
                pool_method=w_eval_selector.pool_method,
                exclude_bos=w_eval_selector.exclude_bos,
                exclude_special_tokens=w_eval_selector.exclude_special_tokens,
            )

    btn_analyze.on_click(do_plot)

    # Display widgets
    display(widgets.VBox([layer1, widgets.HTML("<hr>"), layer2]))


analyze_layerwise_pca()

In [ ]:
def analyze_layerwise_score():
    state = {"models": {}}

    # Define layer 1 widgets for extracting activations
    layer1 = ActivationsExtractorWidget(
        path="../activations",
        dataset="xstest",
    )

    # Define layer 2 widgets for analyzing activations
    w_train_selector = ActivationsSelectorWidget()
    w_eval_selector = ActivationsSelectorWidget()
    btn_analyze = widgets.Button(description="Analyze", button_style="primary", disabled=True)
    out_analyze = widgets.Output()

    w_method = widgets.Dropdown(
        options=["difference_in_mean", "linear_probe"], value="difference_in_mean", description="Method"
    )
    col_train = widgets.VBox([widgets.HTML("<b>Train</b>"), w_train_selector])
    col_eval = widgets.VBox([widgets.HTML("<b>Visualization</b>"), w_eval_selector])
    layer2 = widgets.VBox(
        [
            w_method,
            widgets.HBox([col_train, col_eval]),
            btn_analyze,
            out_analyze,
        ]
    )

    # Register handlers for layer 1
    def update_layer2_widgets(*args):
        activations = layer1.activations

        if activations is None:
            btn_analyze.disabled = True
        else:
            w_train_selector.set_activations(activations)
            w_eval_selector.set_activations(activations)
            btn_analyze.disabled = False
        out_analyze.clear_output()

    layer1.observe(update_layer2_widgets, names="value")

    # Register handlers for layer 2
    def do_plot(*args):
        activations = layer1.activations
        if activations is None:
            return

        with out_analyze:
            out_analyze.clear_output(wait=True)

            # Select samples training
            traub_sample_ids = w_train_selector.samples
            train_activations = activations.select(traub_sample_ids) if traub_sample_ids else activations
            # Train and cache model
            cache_key = (
                tuple(sorted(traub_sample_ids)),
                w_method.value,
                w_train_selector.pool_method,
                w_train_selector.exclude_bos,
                w_train_selector.exclude_special_tokens,
            )
            if cache_key not in state["models"]:
                state["models"][cache_key] = compute_layerwise_score(
                    train_activations,
                    method=w_method.value,
                    pool_method=w_train_selector.pool_method,
                    exclude_bos=w_train_selector.exclude_bos,
                    exclude_special_tokens=w_train_selector.exclude_special_tokens,
                )

            # Select samples for evaluation
            eval_sample_ids = w_eval_selector.samples
            eval_activations = activations.select(eval_sample_ids) if eval_sample_ids else activations
            # Plot evaluation
            plot_layerwise_score(
                eval_activations,
                state["models"][cache_key],
                pool_method=w_eval_selector.pool_method,
                exclude_bos=w_eval_selector.exclude_bos,
                exclude_special_tokens=w_eval_selector.exclude_special_tokens,
            )

    btn_analyze.on_click(do_plot)

    # Display widgets
    display(widgets.VBox([layer1, widgets.HTML("<hr>"), layer2]))


analyze_layerwise_score()